In [10]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

In [11]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [12]:
# Step 1 :- Tools Creation
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()

In [13]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather for a city"""
    url = f"https://api.weatherstack.com/current?access_key=39950d2052867209eb8ce56f481086cb&query={city}"
    response = requests.get(url)
    data = response.json()
    return str(data)

In [14]:
import langchain
import langchain_core

print(langchain.__version__)
print(langchain_core.__version__)

0.1.16
0.1.52


In [15]:
# Step-2 :- Pull the ReAct prompt from Langchain Hub
from langchain.agents.react.agent import create_react_agent
from langchain.agents import AgentExecutor
from langchain import hub
prompt = hub.pull("hwchase17/react")

c:\Users\HEMANTA GHOSH\Desktop\Langchain\venv\lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [16]:
# step-3:- Create the ReAct agent with pulled prompt
agent = create_react_agent(
    llm = llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
)

In [17]:
# step-4:- Wrap it wit agent Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True
)

In [19]:
response = agent_executor.invoke(
    {"input": "Kolkata's current weather"}
)
print(response["output"])



> Entering new AgentExecutor chain...
Thought: To find Kolkata's current weather, I can use a weather API or search for it online.

Action: get_weather_data
Action Input: Kolkata{'success': False, 'error': {'code': 615, 'type': 'request_failed', 'info': 'Your API request failed. Please try again or contact support.'}}Thought: The current weather API request failed. It might be due to a temporary issue with the API or my access to it. Let me try a different approach.

Action: duckduckgo_search
Action Input: "current weather in Kolkata"

RatelimitException: https://links.duckduckgo.com/d.js?q=current+weather+in+Kolkata&kl=wt-wt&l=wt-wt&p=&s=0&df=y&vqd=4-20901793406990356802581802958658555305&ex=-1 202 Ratelimit